# Индексный арбитраж на Московской бирже (MOEX)
### Получение данных, построение непрерывного фьючерса, расчет следящей корзины, бэктест и LIVE-монитор арбитражных окон

Этот ноутбук предназначен для запуска в **Google Colab**. Он решает следующие задачи:
1. Установка необходимых библиотек и импорт зависимостей.
2. Получение исторических дневных данных (OHLCV) по акциям первого эшелона (`SBER`, `GAZP`, `LKOH`, `GMKN`, `NVTK`) через API Московской биржи (ISS MOEX) с 01 января 2020 года.
3. Получение данных по базовому Индексу МосБиржи (`IMOEX`).
4. Получение данных по всем квартальным фьючерсам на Индекс МосБиржи (`MIX`) с 2020 по 2025 гг. с делением цен на 100 (для выравнивания шкал).
5. Сшивка квартальных фьючерсов в единый непрерывный контракт (по максимальному объёму торгов в день).
6. Расчет базиса (спреда) между фьючерсом и индексом, а также построение простейшей следящей корзины (Index Tracking Basket) из топ-5 акций.
7. **Бэктест арбитражной стратегии Cash-and-Carry** со стартовым капиталом 1 000 000 рублей, учетом комиссий, ведением подробного журнала сделок (Trade Log) и расчетом метрик доходности и риска.
8. Интерактивная визуализация результатов (динамика портфеля и точки входа/выхода на графике базиса).
9. **ОНЛАЙН LIVE-МОНИТОР & РОБОТ-СОВЕТНИК** (с защитой от зависания и автоопределением контрактов), который в реальном времени обращается к биржевому серверу, рассчитывает текущие котировки, стаканы, объемы и дает четкие сигналы — что и в каком объеме покупать/продавать.

In [ ]:
# Шаг 1. Установка библиотек
!pip install apimoex pandas plotly scipy requests

In [ ]:
# Шаг 2. Импорт библиотек
import requests
import apimoex
import pandas as pd
import numpy as np
import datetime
import time
from IPython.display import clear_output, display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize

print("Библиотеки успешно импортированы.")

## Получение данных с Московской Биржи (ISS MOEX)
Для получения исторических данных мы будем использовать официальный API Московской Биржи через обертку `apimoex`.

In [ ]:
# Шаг 3. Определение функций получения данных

def get_stock_data(session, ticker, start_date='2020-01-01'):
    """
    Получает дневную историю по акциям с рынка TQBR (основной режим торгов акций)
    """
    print(f"Скачивание акций: {ticker}...")
    columns = ('TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME')
    data = apimoex.get_board_history(session, ticker, board='TQBR', market='shares', engine='stock', columns=columns)
    df = pd.DataFrame(data)
    if df.empty:
        return pd.DataFrame()
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    df = df[df['TRADEDATE'] >= pd.to_datetime(start_date)]
    df = df.set_index('TRADEDATE').sort_index()
    # Преобразуем числовые столбцы
    for col in ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def get_index_data(session, index_ticker='IMOEX', start_date='2020-01-01'):
    """
    Получает дневную историю индекса МосБиржи
    """
    print(f"Скачивание индекса: {index_ticker}...")
    columns = ('TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE')
    data = apimoex.get_board_history(session, index_ticker, board='SNDX', market='index', engine='stock', columns=columns)
    df = pd.DataFrame(data)
    if df.empty:
        return pd.DataFrame()
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    df = df[df['TRADEDATE'] >= pd.to_datetime(start_date)]
    df = df.set_index('TRADEDATE').sort_index()
    for col in ['OPEN', 'HIGH', 'LOW', 'CLOSE']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def get_future_data(session, contract_code, start_date='2020-01-01'):
    """
    Получает дневную историю фьючерсного контракта со срочного рынка FORTS
    Для фьючерса на Индекс (MIX) цены делятся на 100, так как на ISS биржи они представлены
    в формате (цена * 100) относительно пунктов индекса.
    """
    columns = ('TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME')
    data = apimoex.get_board_history(session, contract_code, board='RFUD', market='forts', engine='futures', columns=columns)
    df = pd.DataFrame(data)
    if df.empty:
        return pd.DataFrame()
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    df = df[df['TRADEDATE'] >= pd.to_datetime(start_date)]
    df['SECID'] = contract_code
    for col in ['OPEN', 'HIGH', 'LOW', 'CLOSE']:
        df[col] = pd.to_numeric(df[col], errors='coerce') / 100.0
    df['VOLUME'] = pd.to_numeric(df['VOLUME'], errors='coerce')
    return df

In [ ]:
# Шаг 4. Загрузка данных акций и индекса
start_date = '2020-01-01'
stock_tickers = ['SBER', 'GAZP', 'LKOH', 'GMKN', 'NVTK']

stocks_ohlc = {}
with requests.Session() as session:
    # Скачиваем индекс
    df_imoex = get_index_data(session, 'IMOEX', start_date)
    
    # Скачиваем акции
    for ticker in stock_tickers:
        stocks_ohlc[ticker] = get_stock_data(session, ticker, start_date)

print("\nСкачивание базовых активов завершено!")

In [ ]:
# Шаг 5. Скачивание всех фьючерсов
years = [0, 1, 2, 3, 4, 5]  # Последняя цифра года: 2020, 2021, 2022, 2023, 2024, 2025
months = ['H', 'M', 'U', 'Z']  # Март, Июнь, Сентябрь, Декабрь

futures_list = []
with requests.Session() as session:
    for yr in years:
        for mn in months:
            contract_code = f"MX{mn}{yr}"
            print(f"Скачивание фьючерса {contract_code}...")
            try:
                df_f = get_future_data(session, contract_code, start_date)
                if not df_f.empty:
                    df_f = df_f.dropna(subset=['CLOSE'])
                    if len(df_f) > 0:
                        futures_list.append(df_f)
            except Exception as e:
                print(f"Ошибка при скачивании {contract_code}: {e}")

print(f"\nВсего успешно скачано фьючерсных контрактов: {len(futures_list)}")

In [ ]:
# Шаг 6. Сшивка контрактов по максимальному объему
if futures_list:
    df_all_futs = pd.concat(futures_list, ignore_index=True)
    df_all_futs['VOLUME'] = df_all_futs['VOLUME'].fillna(0)
    df_all_futs = df_all_futs.sort_values(by=['TRADEDATE', 'VOLUME'], ascending=[True, False])
    df_continuous_fut = df_all_futs.drop_duplicates(subset=['TRADEDATE'], keep='first').copy()
    df_continuous_fut = df_continuous_fut.set_index('TRADEDATE').sort_index()
    print(f"Непрерывный фьючерс построен! Всего дней: {len(df_continuous_fut)}")
else:
    print("Данные фьючерсов отсутствуют!")

In [ ]:
# Шаг 7. Объединение и расчет показателей
df_analysis = pd.DataFrame(index=df_imoex.index)
df_analysis['IMOEX'] = df_imoex['CLOSE']
df_analysis['FUTURE'] = df_continuous_fut['CLOSE']
df_analysis['FUTURE_TICKER'] = df_continuous_fut['SECID']

for ticker, df_s in stocks_ohlc.items():
    df_analysis[ticker] = df_s['CLOSE']

df_analysis = df_analysis.dropna(subset=['IMOEX', 'FUTURE'])
df_analysis[stock_tickers] = df_analysis[stock_tickers].ffill().bfill()

df_analysis['BASIS'] = df_analysis['FUTURE'] - df_analysis['IMOEX']
df_analysis['BASIS_PCT'] = (df_analysis['FUTURE'] / df_analysis['IMOEX'] - 1) * 100

In [ ]:
# Шаг 8. Оптимизация следящего портфеля (МНК)
X = df_analysis[stock_tickers].values
y = df_analysis['IMOEX'].values

def tracking_error(weights):
    portfolio_value = np.dot(X, weights)
    return np.mean((y - portfolio_value) ** 2)

initial_weights = np.array([y[0] / (len(stock_tickers) * X[0, i]) for i in range(len(stock_tickers))])
bounds = [(0, None) for _ in range(len(stock_tickers))]

res = minimize(tracking_error, initial_weights, bounds=bounds, method='L-BFGS-B')
best_weights = res.x

df_analysis['BASKET'] = np.dot(X, best_weights)
df_analysis['SPREAD'] = (df_analysis['FUTURE'] / df_analysis['BASKET'] - 1) * 100

## Бэктест Арбитражной Стратегии (Cash-and-Carry)

In [ ]:
# Шаг 9. Расчет сигналов и запуск Бэктеста

# Рассчитываем скользящие метрики на окне 60 дней
window = 60
df_analysis['ROLL_MEAN'] = df_analysis['SPREAD'].rolling(window).mean()
df_analysis['ROLL_STD'] = df_analysis['SPREAD'].rolling(window).std()

# Очищаем начальные строки, где скользящие метрики еще не посчитались
df_bt = df_analysis.dropna(subset=['ROLL_MEAN', 'ROLL_STD']).copy()

# Инициализация бэктеста
initial_capital = 1000000.0
cash = initial_capital
in_position = False
qty = 0.0
basket_entry_price = 0.0
future_entry_price = 0.0
entry_date = None
fee_rate = 0.0005  # 0.05%

trade_log = []
equity_curve = []
dates_list = []

for idx, row in df_bt.iterrows():
    current_date = idx
    spread = row['SPREAD']
    mean_spread = row['ROLL_MEAN']
    std_spread = row['ROLL_STD']
    
    basket_price = row['BASKET']
    future_price = row['FUTURE']
    future_ticker = row['FUTURE_TICKER']
    
    # Расчет текущей стоимости портфеля (Equity / NAV)
    if in_position:
        current_equity = cash + qty * basket_price + qty * (future_entry_price - future_price)
    else:
        current_equity = cash
        
    equity_curve.append(current_equity)
    dates_list.append(current_date)
    
    upper_band = mean_spread + 1.5 * std_spread
    lower_band = mean_spread
    
    if not in_position:
        # Сигнал на ВХОД: спред слишком высок, фьючерс переоценен
        if spread > upper_band and std_spread > 0:
            in_position = True
            entry_date = current_date
            allocated_cash = current_equity * 0.95
            qty = allocated_cash / basket_price
            
            basket_entry_price = basket_price
            future_entry_price = future_price
            
            basket_fee = qty * basket_price * fee_rate
            future_fee = qty * future_price * fee_rate
            total_fee = basket_fee + future_fee
            
            cash = cash - (qty * basket_price) - total_fee
            
            trade_log.append({
                'Тип Сделки': 'ВХОД (ОТКРЫТИЕ АРБИТРАЖА)',
                'Дата': current_date.strftime('%Y-%m-%d'),
                'Фьючерс': future_ticker,
                'Цена Корзины': round(basket_price, 2),
                'Цена Фьючерса': round(future_price, 2),
                'Количество': round(qty, 4),
                'Спред %': round(spread, 3),
                'Комиссия (руб)': round(total_fee, 2),
                'Капитал (руб)': round(current_equity, 2),
                'Причина': 'Спред превысил скользящий верхний порог'
            })
    else:
        # Сигнал на ВЫХОД: спред вернулся к среднему значению
        if spread <= lower_band:
            in_position = False
            
            basket_exit_value = qty * basket_price
            future_pnl = qty * (future_entry_price - future_price)
            
            basket_fee = qty * basket_price * fee_rate
            future_fee = qty * future_price * fee_rate
            total_fee = basket_fee + future_fee
            
            cash = cash + basket_exit_value + future_pnl - total_fee
            
            entry_fees = qty * basket_entry_price * fee_rate + qty * future_entry_price * fee_rate
            trade_pnl = (basket_price - basket_entry_price) * qty + (future_entry_price - future_price) * qty - total_fee - entry_fees
            
            trade_log.append({
                'Тип Сделки': 'ВЫХОД (ЗАКРЫТИЕ АРБИТРАЖА)',
                'Дата': current_date.strftime('%Y-%m-%d'),
                'Фьючерс': future_ticker,
                'Цена Корзины': round(basket_price, 2),
                'Цена Фьючерса': round(future_price, 2),
                'Количество': round(qty, 4),
                'Спред %': round(spread, 3),
                'Комиссия (руб)': round(total_fee, 2),
                'Капитал (руб)': round(cash, 2),
                'Причина': f'Схождение спреда к среднему. Фикс. результат: {round(trade_pnl, 2)} руб'
            })
            
            qty = 0.0
            basket_entry_price = 0.0
            future_entry_price = 0.0
            entry_date = None

# Сохраняем кривую капитала в DataFrame
df_bt['EQUITY'] = equity_curve
df_trade_log = pd.DataFrame(trade_log)

print("Бэктест успешно завершен!")

In [ ]:
# Шаг 10. Расчет метрик качества стратегии
final_equity = df_bt['EQUITY'].iloc[-1]
total_return_pct = (final_equity / initial_capital - 1) * 100

num_years = (df_bt.index[-1] - df_bt.index[0]).days / 365.25
cagr = ((final_equity / initial_capital) ** (1 / num_years) - 1) * 100

df_bt['PEAK'] = df_bt['EQUITY'].cummax()
df_bt['DRAWDOWN'] = (df_bt['EQUITY'] / df_bt['PEAK'] - 1) * 100
max_drawdown = df_bt['DRAWDOWN'].min()

df_bt['DAILY_RET'] = df_bt['EQUITY'].pct_change()
daily_vol = df_bt['DAILY_RET'].std()
sharpe_ratio = (df_bt['DAILY_RET'].mean() / daily_vol) * np.sqrt(252) if daily_vol > 0 else 0

print("=========================================================")
print("                  ФИНАНСОВЫЙ ОТЧЕТ                       ")
print("=========================================================")
print(f"Начальный капитал:       {initial_capital:,.2f} руб.")
print(f"Конечный капитал:        {final_equity:,.2f} руб.")
print(f"Общая доходность:        {total_return_pct:.2f}%")
print(f"Среднегодовая дох. (CAGR): {cagr:.2f}%")
print(f"Максимальная просадка:   {max_drawdown:.2f}%")
print(f"Коэффициент Шарпа:       {sharpe_ratio:.2f}")
print(f"Количество сделок:       {len(df_trade_log)} (входов + выходов)")
print("=========================================================")

In [ ]:
# Вывод таблицы журнала сделок
display(df_trade_log)

In [ ]:
# Шаг 11. Построение графиков бэктеста
fig_bt = make_subplots(rows=2, cols=1, shared_xaxes=True,
                       subplot_titles=('Динамика Капитала (Стартовый капитал: 1 000 000 руб.)', 
                                       'Сигналы на графике спреда (Вход - Красный, Выход - Зеленый)'),
                       vertical_spacing=0.12)

fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt['EQUITY'], name='Баланс счета (NAV)', line=dict(color='green', width=2)), row=1, col=1)

fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt['SPREAD'], name='Текущий спред %', line=dict(color='purple', width=1)), row=2, col=1)
fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt['ROLL_MEAN'], name='Средний спред', line=dict(color='gray', dash='dash')), row=2, col=1)
fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt['ROLL_MEAN'] + 1.5*df_bt['ROLL_STD'], name='Верхняя граница (+1.5 STD)', line=dict(color='orange', dash='dot')), row=2, col=1)

df_opens = df_trade_log[df_trade_log['Тип Сделки'].str.contains('ВХОД')]
df_closes = df_trade_log[df_trade_log['Тип Сделки'].str.contains('ВЫХОД')]

fig_bt.add_trace(go.Scatter(
    x=pd.to_datetime(df_opens['Дата']),
    y=df_opens['Спред %'],
    mode='markers',
    marker=dict(color='red', size=10, symbol='triangle-up'),
    name='Открытие позиции (Вход)'
), row=2, col=1)

fig_bt.add_trace(go.Scatter(
    x=pd.to_datetime(df_closes['Дата']),
    y=df_closes['Спред %'],
    mode='markers',
    marker=dict(color='forestgreen', size=10, symbol='triangle-down'),
    name='Закрытие позиции (Выход)'
), row=2, col=1)

fig_bt.update_layout(
    title_text="Результаты бэктеста индексного арбитража на MOEX",
    height=800,
    template="plotly_white",
    showlegend=True
)

fig_bt.show()

## LIVE Онлайн Монитор & Робот-Советник
Этот скрипт каждые 3 секунды опрашивает серверы Московской биржи (MOEX ISS API), получает текущие стаканы и объёмы торгов для акций и активного фьючерса, рассчитывает спред в режиме реального времени и даёт чёткий совет — **что и в каком объеме покупать/продавать** прямо сейчас.

In [ ]:
# Шаг 12. LIVE Монитор МосБиржи

weights_dict = dict(zip(stock_tickers, best_weights))

latest_mean = df_bt['ROLL_MEAN'].iloc[-1]
latest_std = df_bt['ROLL_STD'].iloc[-1]
live_upper_band = latest_mean + 1.5 * latest_std
live_lower_band = latest_mean

live_capital = 1000000.0

def find_active_future_contract(session):
    """Находит текущий активный торгуемый контракт фьючерса на индекс МосБиржи"""
    url = 'https://iss.moex.com/iss/engines/futures/markets/forts/securities.json'
    r = session.get(url)
    data = r.json()
    cols = data['securities']['columns']
    rows = data['securities']['data']
    df = pd.DataFrame(rows, columns=cols)
    
    mix_futs = df[df['ASSETCODE'] == 'MIX'].copy()
    if mix_futs.empty:
        return 'MXH6'
        
    mix_futs['LASTTRADEDATE'] = pd.to_datetime(mix_futs['LASTTRADEDATE'])
    today = pd.to_datetime(datetime.date.today())
    
    # Ищем контракты, у которых дата экспирации сегодня или в будущем
    active_futs = mix_futs[mix_futs['LASTTRADEDATE'] >= today]
    if active_futs.empty:
        best_contract = mix_futs.sort_values(by='LASTTRADEDATE', ascending=False).iloc[0]['SECID']
    else:
        best_contract = active_futs.sort_values(by='LASTTRADEDATE', ascending=True).iloc[0]['SECID']
        
    return best_contract

def get_live_stock_price(session, ticker):
    """Получает текущие BID, OFFER, LAST цену и дневной объем по акции с резервными фоллбэками"""
    url = f'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}.json'
    r = session.get(url)
    data = r.json()
    cols = data['marketdata']['columns']
    rows = data['marketdata']['data']
    
    # Получаем цену закрытия на случай закрытой биржи
    sec_cols = data['securities']['columns']
    sec_rows = data['securities']['data']
    prev_close = None
    if len(sec_rows) > 0:
        sec_dict = dict(zip(sec_cols, sec_rows[0]))
        prev_close = sec_dict.get('PREVPRICE') or sec_dict.get('LCLOSEPRICE')
        
    if len(rows) > 0:
        row_dict = dict(zip(cols, rows[0]))
        bid = row_dict.get('BID')
        offer = row_dict.get('OFFER')
        last = row_dict.get('LAST') or row_dict.get('LCURRENTPRICE') or prev_close
        return {
            'BID': bid if bid else last,
            'OFFER': offer if offer else last,
            'LAST': last,
            'VOLTODAY': row_dict.get('VOLTODAY', 0) or 0
        }
    return None

def get_live_future_price(session, contract_code):
    """Получает текущие BID, OFFER, LAST цену и дневной объем по фьючерсу"""
    url = f'https://iss.moex.com/iss/engines/futures/markets/forts/securities/{contract_code}.json'
    r = session.get(url)
    data = r.json()
    cols = data['marketdata']['columns']
    rows = data['marketdata']['data']
    
    # Получаем расчетную цену на случай закрытой биржи
    sec_cols = data['securities']['columns']
    sec_rows = data['securities']['data']
    prev_price = None
    if len(sec_rows) > 0:
        sec_dict = dict(zip(sec_cols, sec_rows[0]))
        prev_price = sec_dict.get('PREVPRICE') or sec_dict.get('LASTSETTLEPRICE')
        
    if len(rows) > 0:
        row_dict = dict(zip(cols, rows[0]))
        bid = row_dict.get('BID')
        offer = row_dict.get('OFFER')
        last = row_dict.get('LAST') or prev_price
        
        return {
            'BID': bid / 100.0 if bid else (last / 100.0 if last else None),
            'OFFER': offer / 100.0 if offer else (last / 100.0 if last else None),
            'LAST': last / 100.0 if last else None,
            'VOLTODAY': row_dict.get('VOLTODAY', 0) or 0
        }
    return None

with requests.Session() as session:
    # 1. Автоматически определяем текущий активный фьючерс
    active_future = find_active_future_contract(session)
    print(f"Автоматически определен текущий торгуемый фьючерс: {active_future}")
    print("\n--- Запуск LIVE Мониторинга --- (для остановки прервите выполнение ячейки)")
    
    # Запускаем цикл отслеживания
    for _ in range(100):
        clear_output(wait=True)
        
        # Получаем котировки акций
        stock_prices = {}
        basket_last = 0.0
        basket_bid = 0.0
        basket_offer = 0.0
        
        for ticker in stock_tickers:
            live_data = get_live_stock_price(session, ticker)
            if live_data:
                stock_prices[ticker] = live_data
                w = weights_dict[ticker]
                basket_last += (live_data['LAST'] or live_data['BID'] or 0) * w
                basket_bid += (live_data['BID'] or 0) * w
                basket_offer += (live_data['OFFER'] or 0) * w
        
        # Получаем котировки фьючерса
        fut_data = get_live_future_price(session, active_future)
        
        if fut_data and basket_last > 0:
            fut_last = fut_data['LAST'] or fut_data['BID'] or 0
            fut_bid = fut_data['BID'] or 0
            fut_offer = fut_data['OFFER'] or 0
            
            live_spread = (fut_last / basket_last - 1) * 100
            
            print("=========================================================================")
            print(f"          ТОРГОВЫЙ ТЕРМИНАЛ ОНЛАЙН-МОНИТОРИНГА MOEX (LIVE)              ")
            print(f"          Время обновления: {datetime.datetime.now().strftime('%H:%M:%S')} (Раз в 3 сек.)            ")
            print("=========================================================================")
            print(f"Текущий контракт фьючерса:     {active_future}")
            print(f"Фьючерс:  LAST = {fut_last:,.2f} | BID = {fut_bid:,.2f} | OFFER = {fut_offer:,.2f} | VOL = {fut_data['VOLTODAY']:,} шт.")
            print(f"Корзина:  LAST = {basket_last:,.2f} | BID = {basket_bid:,.2f} | OFFER = {basket_offer:,.2f}")
            print("-------------------------------------------------------------------------")
            print(f"ТЕКУЩИЙ LIVE СПРЕД:            {live_spread:.4f} %")
            print(f"Пороги торговли сегодня:       Средний = {latest_mean:.4f} % | Верхний (+1.5 STD) = {live_upper_band:.4f} %")
            print("=========================================================================")
            print("                       ТЕКУЩИЕ СИГНАЛЫ И СОВЕТЫ                          ")
            print("=========================================================================")
            
            if live_spread > live_upper_band:
                allocated = live_capital * 0.95
                total_basket_units = allocated / basket_last
                
                print("🔥 СИГНАЛ: ОТКРЫТИЕ АРБИТРАЖА (ФЬЮЧЕРС ПЕРЕОЦЕНЕН)!")
                print("👉 СОВЕТ: ПРОДАТЬ ФЬЮЧЕРС И КУПИТЬ КОРЗИНУ АКЦИЙ:")
                print(f"   1. ПРОДАТЬ (ШОРТ) ФЬЮЧЕРС {active_future}: {round(total_basket_units, 2)} контр.")
                print(f"   2. КУПИТЬ АКЦИИ:")
                for ticker in stock_tickers:
                    qty_shares = total_basket_units * weights_dict[ticker]
                    lot_info = "(1 лот = 10 акций)" if ticker == 'SBER' else ""
                    print(f"      - {ticker}: {round(qty_shares, 2)} шт. {lot_info}")
            elif live_spread <= live_lower_band:
                print("✅ СИГНАЛ: СХОЖДЕНИЕ СПРЕДА!")
                print("👉 СОВЕТ: ЗАКРЫТЬ ВСЕ АРБИТРАЖНЫЕ ПОЗИЦИИ (Фиксация прибыли!)")
                print("   1. КУПИТЬ (ВЫКУПИТЬ) ФЬЮЧЕРС")
                print("   2. ПРОДАТЬ КОРЗИНУ АКЦИЙ")
            else:
                print("💤 БЕЗДЕЙСТВИЕ (Спред находится внутри нормальных границ)")
                print("👉 СОВЕТ: Позиции удерживать / Сделок не совершать. Ждем расширения спреда.")
            
            print("=========================================================================")
            print("Детальные объемы торгов (VOLTODAY) по акциям в составе корзины:")
            for ticker in stock_tickers:
                vol = stock_prices[ticker]['VOLTODAY']
                last_p = stock_prices[ticker]['LAST']
                print(f"  - {ticker}: Цена: {last_p:.2f} руб. | Объем торгов за сегодня: {vol:,} шт.")
            
        else:
            print("Ошибка при получении данных с сервера MOEX. Проверка связи...")
            
        time.sleep(3)